In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Definition} $$

$$ 
\mathbb{R}^n \times \mathbb{R}^n \to \mathbb{R}, \quad L(z, t) = 
\frac{1}{N} || z - t ||^2 
$$

$$ \text{Derivative} $$

$$ \frac{dL}{dz_i} = \frac{2}{N} (z_i - t_i) $$

$$ \frac{dL}{dz} = \frac{2}{N} (z - t) $$

$$ \text{Jacobian} $$

$$
J_L(z) =
\nabla _L(z) \top =
\begin{bmatrix}
    \frac{dL}{dz_1}, &
    \frac{dL}{dz_2},  &
    \cdots, &
    \frac{dL}{dz_n}
\end{bmatrix}
$$

$$ dL = J_L(z) \, dz $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dz}, dz \right \rangle =
\left \langle \frac{dF}{dL}, dL \right \rangle \\

\\[2em]

\left \langle \frac{dF}{dL}, dL \right \rangle = \\
\left \langle \frac{dF}{dL}, J_L(z) \, dz \right \rangle = \\
\left \langle {J_L(z)}^\top \, \frac{dF}{dL}, dz \right \rangle \implies \\[1em]

\frac{dF}{dz} = {J_L(z)}^\top \, \frac{dF}{dL} \\[0.5em]
\frac{dF}{dz} = \frac{dL}{dz} \odot \frac{dF}{dL}
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore


def mse(p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
    """Computes the mean squared error between predictions `p` and targets `t`."""
 
    return ((p - t) ** 2).mean()


class MeanSquaredErrorFunction(autograd.Function):
    """Custom autograd function for mean squared error."""

    @staticmethod
    def forward(ctx, z: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        ctx.save_for_backward(z, t)
        return mse(z, t)

    @staticmethod
    def backward(ctx, dF_dL: tr.Tensor) -> tuple[tr.Tensor, tr.Tensor]:
        (z, t) = ctx.saved_tensors
        S = z.shape[0]
        FO = z.shape[1]
        N = S * FO

        dL_dz = 2 / N * (z - t)
        dF_dz = dL_dz * dF_dL

        return (dF_dz, None)


class MeanSquaredError(nn.Module):
    """Mean squared error loss module."""

    def __init__(self):
        super().__init__()

    def forward(self, z: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        return MeanSquaredErrorFunction.apply(z, t)


def test_basic():
  z = tr.tensor([0.2, -1.0, 3.0, 2.5], dtype=tr.float32)
  t = tr.tensor([0.0, -2.0, 1.0, 2.0], dtype=tr.float32)

  actual = mse(z, t)
  expected = tr.nn.MSELoss()(z, t)

  assert actual == approx(expected)


def test_gradcheck(samples) -> None:
    def fn(z, t):
        return MeanSquaredErrorFunction.apply(z, t)

    z = tr.randn((samples, 1), dtype=tr.float64, requires_grad=True)
    t = tr.randn((samples, 1), dtype=tr.float64, requires_grad=False)

    assert autograd.gradcheck(fn, (z, t), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    z = tr.randn(samples, dtype=tr.float32, requires_grad=True)
    t = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    mse_custom = MeanSquaredErrorFunction.apply(z, t)
    mse_builtin = tr.nn.MSELoss()(z, t)

    assert mse_custom == approx(mse_builtin)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)